[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vvvvvivekkk/AI-Engineer/blob/main/02-python-for-data/notebooks/02-pandas-loading-and-cleaning.ipynb)

# 02 · pandas: loading, exploring and cleaning data

**What you will learn**
- load CSV and Excel files into a **DataFrame** (a table)
- the five first-look questions: `shape`, `head`, `info`, `describe`, `isna().sum()`
- select columns, filter rows, `loc` / `iloc`, sort, add columns, dates
- clean a messy file: duplicates, missing values, messy text, numbers stored as text, impossible values
- save the cleaned result

### Finding the practice data

The practice files live in the repo folders `data/` (CSV) and `excel/` (Excel).
The helper below looks for them one folder up from this notebook (that is where they are when you
run the notebook from the repo). If they are not there (for example in Google Colab), it downloads
them from GitHub into the current folder. You do not need to change anything.

In [1]:
from pathlib import Path
import urllib.request

RAW_BASE = "https://raw.githubusercontent.com/vvvvvivekkk/AI-Engineer/main/02-python-for-data/"

def data_path(rel_path):
    """Return a local path to a practice file such as 'data/sales.csv'."""
    for candidate in (Path("..") / rel_path, Path(rel_path)):
        if candidate.exists():
            return candidate
    # Not found locally -> download it from GitHub (works in Colab).
    target = Path(rel_path)
    target.parent.mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve(RAW_BASE + rel_path, target)
    return target

print(data_path("data/sales.csv"))

../data/sales.csv


## 1. Import pandas and load a CSV

`pd.read_csv` reads a CSV file into a DataFrame. `parse_dates=["order_date"]` turns that column into real dates instead of text.

In [2]:
import pandas as pd
print("pandas version:", pd.__version__)

sales = pd.read_csv(data_path("data/sales.csv"), parse_dates=["order_date"])
sales.head()

pandas version: 3.0.5


,order_id,order_date,customer_id,product_id,quantity,unit_price,discount_pct,channel
0,O10001,2026-01-01,C017,P004,3,249,0,Store
1,O10002,2026-01-01,C036,P002,2,2499,0,Website
2,O10003,2026-01-01,C013,P005,6,149,0,Website
3,O10004,2026-01-02,C013,P004,2,249,15,App
4,O10005,2026-01-02,C008,P005,5,149,5,Website


## 2. The five first-look questions

Always run these on a new table. Here are the first two: size and the column types.

In [3]:
print("shape (rows, columns):", sales.shape)
sales.info()

shape (rows, columns): (600, 8)
<class 'pandas.DataFrame'>
RangeIndex: 600 entries, 0 to 599
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   order_id      600 non-null    str           
 1   order_date    600 non-null    datetime64[us]
 2   customer_id   600 non-null    str           
 3   product_id    600 non-null    str           
 4   quantity      600 non-null    int64         
 5   unit_price    600 non-null    int64         
 6   discount_pct  600 non-null    int64         
 7   channel       600 non-null    str           
dtypes: datetime64[us](1), int64(3), str(4)
memory usage: 48.9 KB


`describe()` gives summary statistics for the number columns: count, mean, std (spread), min, quartiles (25 %, 50 % = median, 75 %) and max.

In [4]:
sales.describe()

,order_date,quantity,unit_price,discount_pct
count,600,600.000000,600.000000,600.000000
mean,2026-04-08 21:57:36,2.646667,784.000000,5.383333
min,2026-01-01 00:00:00,1.000000,149.000000,0.000000
25%,2026-02-22 00:00:00,2.000000,249.000000,0.000000
50%,2026-04-13 12:00:00,2.000000,599.000000,5.000000
75%,2026-05-28 00:00:00,4.000000,1199.000000,10.000000
max,2026-06-30 00:00:00,9.000000,2499.000000,20.000000
std,NaN,1.423094,614.114724,6.109987


And how many blanks are in each column? `isna()` marks every missing cell True; `.sum()` counts the Trues per column.

In [5]:
sales.isna().sum()

order_id        0
order_date      0
customer_id     0
product_id      0
quantity        0
unit_price      0
discount_pct    0
channel         0
dtype: int64

## 3. Selecting columns

One name in brackets gives a **Series** (one column). A list of names (double brackets) gives a **DataFrame**.

In [6]:
print(type(sales["quantity"]))
sales[["order_id", "product_id", "quantity"]].head(3)

<class 'pandas.Series'>


,order_id,product_id,quantity
0,O10001,P004,3
1,O10002,P002,2
2,O10003,P005,6


## 4. Filtering rows with conditions

A condition creates a True/False mask, exactly like NumPy. Combine conditions with `&` (and) and `|` (or), and wrap each one in brackets.

In [7]:
big_orders = sales[sales["quantity"] >= 5]
print("orders with 5+ items:", len(big_orders))

app_discounted = sales[(sales["channel"] == "App") & (sales["discount_pct"] > 0)]
print("discounted app orders:", len(app_discounted))

store_or_app = sales[sales["channel"].isin(["Store", "App"])]
print("store or app orders:", len(store_or_app))

orders with 5+ items: 67
discounted app orders: 109
store or app orders: 309


## 5. `loc` (by label) and `iloc` (by position)

- `loc[rows, columns]` uses names and conditions.
- `iloc[rows, columns]` uses integer positions, like NumPy slicing.

In [8]:
print(sales.loc[sales["discount_pct"] == 20, ["order_id", "unit_price", "discount_pct"]].head(3))
print()
print(sales.iloc[0:3, 0:4])

   order_id  unit_price  discount_pct
21   O10022         149            20
32   O10033         249            20
40   O10041        1199            20

  order_id order_date customer_id product_id
0   O10001 2026-01-01        C017       P004
1   O10002 2026-01-01        C036       P002
2   O10003 2026-01-01        C013       P005


## 6. Adding a column and sorting

Revenue for each order = quantity × unit price × (1 − discount ÷ 100). pandas computes it for all rows at once.

In [9]:
sales["revenue"] = sales["quantity"] * sales["unit_price"] * (1 - sales["discount_pct"] / 100)
sales.sort_values("revenue", ascending=False).head(5)

,order_id,order_date,customer_id,product_id,quantity,unit_price,discount_pct,channel,revenue
215,O10216,2026-03-15,C008,P002,6,2499,0,Website,14994.0
509,O10510,2026-06-10,C057,P002,5,2499,0,Website,12495.0
234,O10235,2026-03-24,C009,P002,4,2499,0,Website,9996.0
293,O10294,2026-04-11,C018,P002,4,2499,5,Website,9496.2
306,O10307,2026-04-15,C056,P012,5,1799,0,Store,8995.0


## 7. Working with dates

Because `order_date` is a real datetime, `.dt` gives us its parts. `to_period("M")` turns a date into its month (like `2026-03`).

In [10]:
sales["month"] = sales["order_date"].dt.to_period("M")
sales["weekday"] = sales["order_date"].dt.day_name()
print(sales[["order_date", "month", "weekday"]].head())
print()
print("first order:", sales["order_date"].min().date(), "| last order:", sales["order_date"].max().date())

  order_date    month   weekday
0 2026-01-01  2026-01  Thursday
1 2026-01-01  2026-01  Thursday
2 2026-01-01  2026-01  Thursday
3 2026-01-02  2026-01    Friday
4 2026-01-02  2026-01    Friday

first order: 2026-01-01 | last order: 2026-06-30


## 8. Counting categories with `value_counts`

`value_counts()` counts each distinct value. It is the fastest way to understand a text column and to spot typos.

In [11]:
sales["channel"].value_counts()

channel
Website    291
App        209
Store      100
Name: count, dtype: int64

## 9. Loading from Excel

The same data also lives in `excel/practice-datasets.xlsx`, one dataset per sheet. `sheet_name=None` loads **every** sheet into a dictionary.

In [12]:
xlsx = data_path("excel/practice-datasets.xlsx")
sheets = pd.read_excel(xlsx, sheet_name=None)
for name, table in sheets.items():
    print(f"{name:12s} {table.shape}")

About        (6, 3)
Sales        (600, 8)
Customers    (60, 5)
Products     (12, 4)
Students     (120, 7)
Messy_Sales  (85, 7)


Load a single sheet by name. The `About` sheet explains what each sheet holds.

In [13]:
about = pd.read_excel(xlsx, sheet_name="About")
about

,Sheet,Rows,What it contains
0,Sales,600.0,"One row per order, Jan-Jun 2026. Revenue = qua..."
1,Customers,60.0,"One row per customer: city, segment (Retail/St..."
2,Products,12.0,12 products in 4 categories with list price in...
3,Students,120.0,"Study hours, attendance and three exam scores...."
4,Messy_Sales,85.0,"A dirty export: duplicates, blanks, mixed-case..."
5,Note,NaN,All data is synthetic (generated by make_data....


# Part 2 · Cleaning a messy file

`sales_messy.csv` is a deliberately dirty export: the kind of file you really receive. We clean it step by step, checking after every step.

In [14]:
messy = pd.read_csv(data_path("data/sales_messy.csv"))
print("shape:", messy.shape)
messy.head(8)

shape: (85, 7)


,order_id,order_date,customer_name,city,product,quantity,price
0,O10079,2026-01-27,Lakshmi Rao,mumbai,USB-C Charger,2.0,"1,199"
1,O10001,2026-01-01,Lakshmi Joshi,New Delhi,a5 notebook (5 pack),3.0,Rs 249
2,O10069,2026-01-25,Varun Joshi,chennai,Steel Water Bottle,4.0,449
3,O10023,2026-01-08,Ananya Menon,Mumbai,STEEL WATER BOTTLE,3.0,449
4,O10013,2026-01-04,Pooja Joshi,NaN,steel water bottle,2.0,Rs 449
5,O10034,2026-01-11,Sneha Sharma,Bangalore,GEL PEN SET,5.0,149
6,O10011,2026-01-03,Rahul Iyer,chennai,Cotton T-Shirt,1.0,Rs 499
7,O10019,2026-01-07,Diya Singh,Bengaluru,Wireless Mouse,3.0,799


## 10. Diagnose before you fix

Look at types, blanks, duplicates and the distinct values of text columns. Notice:
- `price` is text (`str`), not a number
- `city` has many spellings of the same city
- some rows are exact duplicates

In [15]:
print(messy.dtypes, "\n")
print("missing per column:\n", messy.isna().sum(), "\n")
print("duplicate rows:", messy.duplicated().sum())
print("\ncity spellings:", sorted(messy["city"].dropna().unique()))

order_id             str
order_date           str
customer_name        str
city                 str
product              str
quantity         float64
price                str
dtype: object 

missing per column:
 order_id         0
order_date       2
customer_name    0
city             3
product          0
quantity         4
price            0
dtype: int64 

duplicate rows: 5

city spellings: [' Hyderabad', 'Bangalore', 'Bengaluru', 'Bengaluru ', 'Chennai', 'HYDERABAD', 'Hyderabad', 'Hyderabad ', 'MUMBAI', 'Mumbai', 'New Delhi', 'Pune', 'bengaluru', 'chennai ', 'delhi', 'hyderabad', 'mumbai', 'pune ']


## 11. Remove duplicates

`drop_duplicates()` keeps the first copy of each repeated row. We count rows before and after so we know what changed.

In [16]:
before = len(messy)
messy = messy.drop_duplicates()
print(f"removed {before - len(messy)} duplicate rows -> {len(messy)} rows left")

removed 5 duplicate rows -> 80 rows left


## 12. Clean the text columns

`.str.strip()` removes spaces at both ends, `.str.title()` makes "hyderabad" into "Hyderabad", and `replace` merges different names for the same city.
We do the same for the product names. Then `value_counts()` confirms the result.

In [17]:
messy["city"] = (messy["city"]
                 .str.strip()
                 .str.title()
                 .replace({"Bangalore": "Bengaluru", "New Delhi": "Delhi"}))
messy["product"] = messy["product"].str.strip().str.title()
messy["city"].value_counts(dropna=False)

city
Bengaluru    24
Mumbai       20
Hyderabad    15
Chennai      10
Delhi         4
Pune          4
NaN           3
Name: count, dtype: int64

## 13. Turn text prices into numbers

Prices look like `"1,299"` or `"Rs 799"`. We delete every character that is not a digit or a dot (a **regular expression** `[^0-9.]` does that), then convert with `astype(float)`.

In [18]:
print("before:", messy["price"].head(6).tolist())
messy["price"] = messy["price"].str.replace(r"[^0-9.]", "", regex=True).astype(float)
print("after: ", messy["price"].head(6).tolist())
print("dtype now:", messy["price"].dtype)

before: ['1,199', 'Rs 249', '449', '449', 'Rs 449', '149']
after:  [1199.0, 249.0, 449.0, 449.0, 449.0, 149.0]
dtype now: float64


## 14. Handle missing values

Decide per column, based on meaning:
- **order_date** missing → we cannot place the order in time, so **drop** those rows.
- **quantity** missing → we fill with the **median** quantity (a typical order), rounded to a whole number because nobody buys half a mouse. Note: we write `messy["quantity"] = messy["quantity"].fillna(...)`; the old `inplace=True` style does not work in pandas 3.
- **city** missing → fill with the label `"Unknown"` so the rows still count in totals.

In [19]:
messy = messy.dropna(subset=["order_date"])
median_qty = int(round(messy["quantity"].median()))
messy["quantity"] = messy["quantity"].fillna(median_qty)
messy["city"] = messy["city"].fillna("Unknown")
print("median quantity used:", median_qty)
messy.isna().sum()

median quantity used: 2


order_id         0
order_date       0
customer_name    0
city             0
product          0
quantity         0
price            0
dtype: int64

## 15. Remove impossible values and fix types

A negative quantity is a typo. We keep only positive quantities, then store quantity as a whole number and the date as a real date.

In [20]:
print("rows with quantity <= 0:")
print(messy[messy["quantity"] <= 0])
messy = messy[messy["quantity"] > 0].copy()
messy["quantity"] = messy["quantity"].astype(int)
messy["order_date"] = pd.to_datetime(messy["order_date"])
messy.dtypes

rows with quantity <= 0:
   order_id  order_date  customer_name       city               product  \
39   O10026  2026-01-09  Lakshmi Gupta  Hyderabad  Bluetooth Headphones   

    quantity   price  
39      -2.0  2499.0  


order_id                    str
order_date       datetime64[us]
customer_name               str
city                        str
product                     str
quantity                  int64
price                   float64
dtype: object

## 16. Check the cleaned table and save it

`.copy()` above makes `messy` its own independent table (good practice after filtering).
We save into an `outputs/` folder (created if missing). The repo's `.gitignore` ignores `outputs/`, so your experiments do not clutter the project.

In [21]:
messy["revenue"] = messy["quantity"] * messy["price"]
print("clean shape:", messy.shape)
print("total revenue in this export: Rs", round(messy["revenue"].sum()))

out = Path("outputs")
out.mkdir(exist_ok=True)
messy.to_csv(out / "sales_clean.csv", index=False)
print("saved", out / "sales_clean.csv")

clean shape: (77, 8)
total revenue in this export: Rs 135779


saved outputs/sales_clean.csv


## Recap

1. Load (`read_csv`, `read_excel`) → 2. look (`shape`, `head`, `info`, `describe`, `isna().sum()`) →
3. clean (duplicates, text, types, missing, impossible values) → 4. save.

Write down every cleaning rule you applied. Next: **03 · groupby, merge and Excel reports**.